# Automated Hyperparameter Optimization and Model Selection for NLP Pipelines

*NLPAICS 2026 Summer School · Ernesto Luis Estevanell*

In the talk we walked through an idea: instead of hand-tuning a model, we can let a search do the work for us, and with AutoGOAL we can search over whole pipelines, not just their settings. This notebook is where we actually do it.

We'll stay with one small text-classification problem the whole way through, the 20 Newsgroups corpus, and get a little more ambitious in stages. First we build and tune a classifier with scikit-learn, the way you've probably done before. Then we hand the whole pipeline over to AutoGOAL and let it build one for us. Finally we teach AutoGOAL something new: we add our own building block, a pretrained transformer, and let the search decide whether to use it.

Run the cells in order. Here and there I'll hand it over to you to change something and run it again. A couple of those are marked *after class*, because they start a longer search or a download; in the session just read past them and come back to them on your own time.

## Getting set up

One cell to confirm the kernel is the right one and AutoGOAL is importable. If it fails, the message tells you how to fix it, almost always *Kernel > Change Kernel > "NLPAICS 07"*.

In [ ]:
# --- Sanity check: right kernel, AutoGOAL present -----------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
# Run fully offline: the model + data are pre-cached by setup.sh, so we never let
# HuggingFace phone home mid-session (a blocking call that stalls on conference wifi).
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
# macOS only: let AutoGOAL's worker processes fork safely (no-op on Linux/vast.ai)
os.environ.setdefault("OBJC_DISABLE_INITIALIZE_FORK_SAFETY", "YES")

if sys.version_info[:2] not in [(3, 9), (3, 10)]:
    raise SystemExit(f"Kernel is Python {sys.version.split()[0]}, AutoGOAL needs 3.9/3.10. "
                     "Fix: Kernel > Change Kernel > 'NLPAICS 07'.")
try:
    import autogoal
    from autogoal_contrib import find_classes
except ModuleNotFoundError as e:
    raise SystemExit(f"AutoGOAL not in this kernel (missing '{e.name}'). "
                     "Pick the 'NLPAICS 07' kernel and make sure ./setup.sh 07 finished.")

# AutoGOAL runs each candidate pipeline in a forked worker process; this line just
# makes that safe and identical on Linux/vast.ai and macOS (you can ignore it).
import multiprocessing as mp
try:
    mp.set_start_method("fork", force=True)
except (RuntimeError, ValueError):
    pass

algos = find_classes()
print("Python", sys.version.split()[0])
print(f"AutoGOAL ready -- {len(algos)} building blocks available.")

## The data: 20 Newsgroups

Everything in this notebook runs on one dataset, so it's worth a minute to meet it.

**20 Newsgroups** is a classic text-classification benchmark: roughly 18,000 posts from late-1990s Usenet discussion groups, each belonging to exactly one newsgroup. We use a **four-topic slice** (`rec.autos`, `sci.med`, `sci.space`, and `talk.politics.misc`), small enough to search live but still a genuine four-way problem.

One preprocessing choice matters a lot here: we strip each post's **headers, footers, and signature quotes**. Those are full of dead giveaways (the newsgroup name often sits right in the header), and leaving them in lets a model reach ~90%+ without reading the actual language. Stripping them forces the model to learn from the *text*, which is the point, and why our numbers are lower (and more honest) than some you'll see quoted elsewhere.

In [ ]:
# --- Load the four-topic slice (cached by setup.sh) ---------------------------
from collections import Counter
from sklearn.datasets import fetch_20newsgroups

CATEGORIES = ['rec.autos', 'sci.med', 'sci.space', 'talk.politics.misc']
strip = ('headers', 'footers', 'quotes')   # drop metadata giveaways; learn the language
train = fetch_20newsgroups(subset='train', categories=CATEGORIES, remove=strip, random_state=0)
test  = fetch_20newsgroups(subset='test',  categories=CATEGORIES, remove=strip, random_state=0)

print(f"train: {len(train.data)} docs    test: {len(test.data)} docs    classes: {len(train.target_names)}")
print("classes:", train.target_names)

# one real example, so we know what a "document" looks like
ex = next(i for i, d in enumerate(train.data) if len(d.strip()) > 60)
print("\n--- an example document (first 300 chars) ---\n")
print(train.data[ex][:300].strip(), "...")
print("\nlabel:", train.target_names[train.target[ex]])

In [ ]:
# --- A quick look: how big, and how balanced? --------------------------------
import numpy as np
import matplotlib.pyplot as plt

names = train.target_names
tr = Counter(train.target); te = Counter(test.target)
print("class balance (train):", {names[i]: tr[i] for i in range(len(names))})

# how long are the posts? (words, after stripping metadata)
lengths = np.array([len(d.split()) for d in train.data])
print(f"document length (words): median {int(np.median(lengths))}, "
      f"mean {lengths.mean():.0f}, max {lengths.max()}  "
      f"({(lengths == 0).sum()} posts went empty after stripping)")

x = np.arange(len(names)); w = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, [tr[i] for i in range(len(names))], w, label=f'train (n={len(train.data)})')
ax.bar(x + w/2, [te[i] for i in range(len(names))], w, label=f'test (n={len(test.data)})')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('documents'); ax.set_title('20 Newsgroups (4 topics): class balance')
ax.legend(); ax.grid(axis='y', alpha=0.3); fig.tight_layout(); plt.show()

### How we'll use the data

20 Newsgroups already comes divided into a training set and a test set, so there is nothing for us to split by hand. What matters is how we use each one, and since the rule stays the same for the whole notebook, I'll say it once, here.

The test set is the final exam. We do not look at it while we work; we open it once, at the end of each part, only to report an honest number.

Everything else, every model and every setting we compare, is judged without ever touching the test set. When we tune by hand (Parts 1 and 2) we reason about the settings and only confirm on the test set at the very end. When a *search* does the choosing (AutoGOAL in Parts 3 and 4, or grid/random search in the appendix), it carves its own validation data out of the training set: scikit-learn's search rotates through cross-validation folds, and AutoGOAL holds back an internal slice to score each pipeline it tries. Either way, the test set stays sealed until the final check.

## Part 1: Tuning a classifier with scikit-learn

Let's start where you've probably been before. A text classifier is two pieces stuck together: a **vectorizer** that turns text into numbers, and a **classifier** that learns from those numbers. Each piece has settings, and choosing good ones is what "tuning" means.

We'll wire the two pieces together by hand with scikit-learn's `fit`/`predict` API, meet **NLTK** (the classic toolkit for the text side), and improve the model just by making a few sensible choices.

*(The systematic way to sweep those settings, grid and random search, lives in an [appendix](#appendix) at the end, so the session can spend its time on the AutoML payoff. We'll point you there at the right moment.)*

In [ ]:
# --- 1.1  The two pieces, wired by hand ---------------------------------------
# scikit-learn estimators all speak the same little API:
#   vectorizer:  .fit_transform(texts) -> matrix   |   .transform(texts) -> matrix
#   classifier:  .fit(matrix, labels)              |   .predict(matrix)  -> labels
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

vec = CountVectorizer()                        # text  -> word-count features
X_train = vec.fit_transform(train.data)        # LEARN the vocabulary on train, then encode
X_test  = vec.transform(test.data)             # encode test with that SAME vocabulary

clf = LinearSVC(max_iter=5000, random_state=0) # a plain linear classifier
clf.fit(X_train, train.target)                 # learn from the training matrix
pred = clf.predict(X_test)                      # predict on the test matrix

print(f"vocabulary learned : {len(vec.vocabulary_):,} words")
print(f"training matrix    : {X_train.shape[0]} docs × {X_train.shape[1]:,} features")
print(f"baseline (CountVectorizer -> LinearSVC), test accuracy = "
      f"{accuracy_score(test.target, pred):.1%}")

So our starting point is **72.1%**, from nothing more than word counts and an off-the-shelf linear SVM. Notice the shape of what we did: it's the same handful of moves for almost any scikit-learn model:

- `fit_transform` / `transform` on the vectorizer, **fit on train only**, so the vocabulary is learned from training data and the test set stays untouched;
- `fit` then `predict` on the classifier.

That `fit` on train, `transform`/`predict` elsewhere discipline is the whole game in miniature: the moment test text leaks into something you `fit`, your number stops being honest.

Before we try to improve it, let's open up that first step. What does "turn text into word counts" actually involve?

In [ ]:
# --- 1.2  What "turn text into numbers" really does: tokenize (meet NLTK) ------
# A vectorizer first splits text into tokens. scikit-learn does this with a simple
# regex; NLTK does it with linguistically-aware tools -- and adds stopword lists and
# stemmers, the classic building blocks of a text pipeline.
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

doc = next(d for d in train.data if 60 < len(d.strip()) < 200)   # a short, real post
stop_en = set(stopwords.words('english'))
stemmer = PorterStemmer()

tokens  = word_tokenize(doc)                                               # text -> words
content = [t for t in tokens if t.isalpha() and t.lower() not in stop_en]  # drop stopwords/punct
stems   = [stemmer.stem(t) for t in content]                               # words -> stems

print("RAW          :", repr(doc.strip()[:120]))
print("NLTK tokens  :", tokens[:14])
print("no stopwords :", content[:12])
print("stemmed      :", stems[:12])
print(f"\n(NLTK ships a {len(stop_en)}-word english stopword list, e.g. {sorted(stop_en)[:6]})")

Three classic moves, all from **NLTK**: split into tokens, drop *stopwords* (high-frequency words like "the", "is", "in" that carry little topic signal), and *stem* words down to a common root ("landing", "landed" become "land") so they collapse into one feature.

scikit-learn's vectorizers already do a lighter version of this internally: that's what `stop_words='english'` and their built-in token pattern are, and you can hand them a custom tokenizer with `TfidfVectorizer(tokenizer=...)`. The point for now: **how you turn text into tokens is itself a choice**, another knob to tune. And these very NLTK steps are what AutoGOAL will later treat as searchable building blocks in Part 3.

In [ ]:
# --- 1.3  Tuning by hand: better choices, same API ----------------------------
# We don't need a search to beat the baseline -- three well-known choices for text
# already help a lot: TF-IDF weighting instead of raw counts, word *pairs* (bigrams)
# on top of single words, and dropping common English stopwords.
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')   # tuned by reasoning
X_train = vec.fit_transform(train.data)
X_test  = vec.transform(test.data)

clf = LinearSVC(C=1, max_iter=5000, random_state=0)
clf.fit(X_train, train.target)
acc = accuracy_score(test.target, clf.predict(X_test))
print(f"hand-tuned (TF-IDF + bigrams + stopwords -> LinearSVC), test accuracy = {acc:.1%}")
print( "   baseline was 72.1% -- a big jump, just from better settings")

From 72.1% to **84.1%**, with no search at all, just three sensible decisions about how to represent the text. That's the upside of hyperparameters: a little domain knowledge goes a long way.

The catch is everything we *didn't* try. There are many more knobs (how many n-grams, min/max document frequency, the SVM's `C`, and more) and far too many combinations to check by hand. Before we automate that, one piece of housekeeping makes everything downstream cleaner.

## Part 2: The scikit-learn `Pipeline` abstraction

Look back at 1.1 and 1.3: each time we did the same dance (`fit_transform` the vectorizer, `transform` the test set, `fit` the classifier, `predict`), juggling the pieces by hand. That's tedious, and it's the easiest place to accidentally leak test data into training.

scikit-learn's **`Pipeline`** packages a sequence of steps into a single estimator. One `.fit()` and one `.predict()` flow through the whole chain: same model, far less to get wrong.

In [ ]:
# --- 2.1  The hand-tuned model as one Pipeline (our bar for AutoGOAL) ----------
from sklearn.pipeline import Pipeline

best_hand_tuned = Pipeline([
    ('vec', TfidfVectorizer(ngram_range=(1, 2), stop_words='english')),
    ('clf', LinearSVC(C=1, max_iter=5000, random_state=0)),
]).fit(train.data, train.target)        # exactly the model from 1.3, now one object

HAND_TUNED_ACC = best_hand_tuned.score(test.data, test.target)
print("pipeline steps:", [name for name, _ in best_hand_tuned.steps])
print(f"hand-tuned Pipeline, test accuracy = {HAND_TUNED_ACC:.1%}   (identical to 1.3)")
print( ">> This is our bar: the number AutoGOAL has to beat.")

Same 84.1%, now a single object, and that buys us one more thing: a clean way to **name** each step's settings. A step name, a double underscore, the hyperparameter: `vec__ngram_range`, `clf__C`.

That naming is the doorway to *systematic* tuning. Instead of trying knobs by hand, you hand scikit-learn a space like `{'vec__ngram_range': [...], 'clf__C': [...]}` and let **grid or random search** sweep it. That's classical HPO, worth knowing, and worked through in the [appendix](#appendix) (it finds essentially this same ~84% config, automatically).

But sweeping the knobs of a *fixed* pipeline is only half the story. We still chose the shape ourselves: a vectorizer, then a classifier, a linear SVM at that. What if the search could choose the steps too? That is the leap from hyperparameter tuning to **AutoML**, and where AutoGOAL comes in next.

## Part 3: Searching over whole pipelines with AutoGOAL

So far *we* fixed the shape of the pipeline (vectorizer then classifier) and tuned it by hand to the ~84% bar we just locked in (`HAND_TUNED_ACC`). Now we turn it around. We hand AutoGOAL a box of building blocks and a time budget, and it assembles the pipeline for us, choosing the steps, their order, and all their settings. The question for this part: **can a search that *builds* the pipeline beat the one we tuned by hand?**

As I mentioned in the talk, AutoGOAL does this with a probabilistic grammar. Picture every valid pipeline as a sentence the grammar can produce; the search gradually learns which sentences tend to score well. The building blocks come from scikit-learn and NLTK, and what keeps them from being glued together nonsensically is a type system that knows what can feed into what. Let's look at those pieces one at a time, and then run a search.

### The building blocks

One idea makes all of this part click: **every block has a type for what it takes in and what it gives out**, basically *what kind of thing goes in, and what comes out*. A vectorizer takes text and returns a matrix of numbers; a classifier takes a matrix and returns labels. AutoGOAL only connects two blocks when those types line up, and that's how it assembles pipelines that make sense.

So first, the menu of blocks. Read the plain-English roles; don't worry about the `Tensor[...]` spelling yet.

In [ ]:
# --- 3.1  The catalogue: what AutoGOAL can use, grouped by what it OUTPUTS --------
# find_classes() is the menu of every block AutoGOAL discovered (wrapped from scikit-
# learn and NLTK). You can ALSO import any of them by name -- e.g.
#     from autogoal_sklearn import TfidfVectorizer, LinearSVC
# and the imported class IS the very same wrapped block find_classes() returns. We'll
# do exactly that below. First, the menu, grouped by the TYPE each block produces --
# which is really "what job does it do".
from collections import defaultdict
from autogoal_contrib import find_classes
import autogoal_sklearn, autogoal_nltk

all_blocks = find_classes()
print(f"{len(all_blocks)} blocks total   |   "
      f"sklearn: {len(find_classes(modules=[autogoal_sklearn]))}   "
      f"nltk: {len(find_classes(modules=[autogoal_nltk]))}\n")

ROLE = {   # a human label for each output type we care about for text classification
    "Tensor[1, Categorical, Dense]": "CLASSIFIERS          (-> a class label)",
    "Tensor[2, Continuous, Sparse]": "VECTORIZERS          (text -> sparse features)",
    "Tensor[2, Continuous, Dense]" : "MATRIX TRANSFORMERS  (reduce / scale features)",
    "Seq[Word]"                    : "TOKENIZERS  [NLTK]   (text -> words)",
    "Stem"                         : "STEMMERS    [NLTK]   (word -> stem)",
}
by_out = defaultdict(list)
for c in all_blocks:
    by_out[str(c.output_type())].append(c.__name__)
for out, role in ROLE.items():
    print(role)
    print("    " + ", ".join(sorted(by_out[out])) + "\n")
print("(The rest -- taggers, chunkers, regressors, clustering -- won't help classify a document.)")

### The task, written in types

We can state the whole job in one line of types. We put in the documents and their labels; we want predicted labels back:

`(Seq[Sentence], Supervised[VectorCategorical]) -> VectorCategorical`

`Seq[Sentence]` is just "a list of texts" and `VectorCategorical` is "class labels". The one subtlety is **`Supervised[...]`**: the labels we hand in for *training* and the labels the model *predicts* are the same kind of thing, so we wrap the training ones as `Supervised` to mark them as "given to us as ground truth", so the search never tries to *produce* them mid-pipeline. Let's pin this down in code.

In [ ]:
# --- 3.2  The vocabulary (imported once) and our task, pinned down ---------------
from autogoal.kb import (
    Seq, Sentence, Word, Document, Text,     # text granularity: a Word is a Sentence is a Document is a Text
    Supervised, VectorCategorical,           # labels in (Supervised) | class labels out
)
# Our task, once and for all -- reused by every search below:
INPUT  = (Seq[Sentence], Supervised[VectorCategorical])
OUTPUT = VectorCategorical
print("TASK:", INPUT, "->", OUTPUT, "\n")

# The friendly names are aliases -- handy when you read AutoGOAL's output:
from autogoal.kb import MatrixContinuousSparse, MatrixContinuousDense
print("legend:  VectorCategorical      ==", repr(VectorCategorical))
print("         MatrixContinuousSparse ==", repr(MatrixContinuousSparse))
print("         MatrixContinuousDense  ==", repr(MatrixContinuousDense))

# A classifier literally advertises the Supervised label as its 2nd input:
from autogoal_sklearn import LinearSVC
print("\nLinearSVC wiring:", LinearSVC.input_types(), "->", LinearSVC.output_type())

### Your box of blocks

The blocks you hand the search **are** the search space. A *registry* is nothing fancy, just a Python list of the blocks you want to allow. Let's import a handful by name and bundle them, then write a tiny helper, `show_registry`, that prints how each block wires up and checks the box can form a valid pipeline *before* we spend time searching.

In [ ]:
# --- 3.3  Curate a box of blocks (a registry) + a validator ----------------------
from autogoal_sklearn import (TfidfVectorizer, CountVectorizer,
                              LinearSVC, LogisticRegression, SGDClassifier,
                              MultinomialNB, ComplementNB)
from autogoal.kb import build_pipeline_graph

registry = [TfidfVectorizer, CountVectorizer, LinearSVC, LogisticRegression,
            SGDClassifier, MultinomialNB, ComplementNB]

def show_registry(reg):
    """Print each block's wiring, and check the box forms at least one valid pipeline."""
    if not reg:
        print("(empty box -- add at least a vectorizer and a classifier)"); return
    for c in reg:
        print(f"  {c.__name__:18s} {c.input_types()}  ->  {c.output_type()}")
    try:
        g = build_pipeline_graph(input_types=INPUT, output_type=OUTPUT, registry=reg)
        print(f"\n  valid search space: {g.graph.number_of_nodes()} nodes, "
              f"{g.graph.number_of_edges()} edges")
    except TypeError:
        print("\n  no valid pipeline from this box -- you need a path in->out "
              "(usually a vectorizer AND a classifier).")

show_registry(registry)

### Letting it search: the payoff

Now we run the search for real. AutoGOAL explores the space of valid pipelines with an evolutionary search it calls **PESearch**: it tries pipelines, scores each on a held-out slice of the training data, and gradually leans toward what works. We hand it our task (`INPUT -> OUTPUT`), the registry, and a **budget**.

Unlike the appendix's grid search (5-fold CV), each pipeline here is scored on a *single* held-out split (`cross_validation_steps=1`) to stay within budget: faster, but a noisier estimate, which is part of why scores wander from run to run.

> **🔮 Predict.** Your hand-tuned pipeline reached ~84%. AutoGOAL starts from scratch and picks the vectorizer *and* the classifier *and* their settings itself. Will it beat 84%, or just match it?
>
> *(While it runs you'll see pipelines stream past for ~90s. Gray error lines are pipelines that failed or timed out; the search just skips them. Watch the best score climb.)*

In [ ]:
# --- 3.4  Run the search (for real) ----------------------------------------------
# AutoGOAL scores each pipeline on a held-out slice and leans toward what works. We pass
# our task, the registry, and a BUDGET. The budget knobs (search_timeout / evaluation_
# timeout / pop_size) are forwarded to PESearch; search_timeout is the anytime "stop", so
# it -- not search_iterations -- ends the run. ConsoleLogger prints a plain, scrolling log.
from autogoal.ml import AutoML
from autogoal.search import ConsoleLogger

y_train = [train.target_names[i] for i in train.target]   # AutoGOAL wants string labels
y_test  = [train.target_names[i] for i in test.target]

automl = AutoML(
    input=INPUT, output=OUTPUT, registry=registry,
    search_iterations=10_000,    # let the CLOCK be the stop, not the generation count
    search_timeout=90,           # total budget -- the anytime stop
    evaluation_timeout=20,       # cap any single pipeline
    pop_size=10,                 # pipelines per generation
    cross_validation_steps=1,    # one held-out split per pipeline (quick; see note above)
    errors="ignore", random_state=0,
)
automl.fit(train.data, y_train, logger=ConsoleLogger())

print("\nbest validation score:", automl.best_scores_)
print("best pipeline AutoGOAL built:\n", automl.best_pipelines_[0])

In [ ]:
# --- 3.4b  Open the test set, once -------------------------------------------
import numpy as np
pred = list(automl.predict(test.data))
acc = np.mean([p == t for p, t in zip(pred, y_test)])
print(f"AutoGOAL best pipeline, TEST accuracy = {acc:.1%}")
if 'HAND_TUNED_ACC' in globals():
    print(f"(our hand-tuned bar was {HAND_TUNED_ACC:.1%})")

Did it beat your prediction? AutoGOAL built a TF-IDF and linear-classifier pipeline and reached **around 86%** on the test set, right in the range of the pipeline we tuned by hand, except this time it chose the vectorizer, the classifier, and every setting itself.

A couple of things to read off the output. `best_scores_` is a list, one entry per pipeline AutoGOAL kept, and the number is its score on AutoGOAL's own internal validation split, so it sits a little above the test score. And the result genuinely **wanders from run to run, even with a fixed random seed**, because the budget is a fixed amount of *time*, not a fixed number of trials: a faster or slower machine fits in more or fewer candidates before the clock runs out. So expect a number in the mid-80s, not my exact one.

### Under the hood: how did it know what to connect? *(optional)*

You just watched AutoGOAL build a winning pipeline. For the curious, here's the machinery that made it possible: a typed *grammar* of pipelines. Feel free to skip; nothing later depends on it.

In [ ]:
# --- 3.5  Define a space, read it, sample from it -----------------------------
from autogoal.grammar import (generate_cfg, Sampler, Union,
                              ContinuousValue, CategoricalValue, BooleanValue, DiscreteValue)
from autogoal.utils import nice_repr

@nice_repr
class SVM:
    def __init__(self, C: ContinuousValue(0.1, 10),
                 kernel: CategoricalValue("linear", "rbf"),
                 shrinking: BooleanValue()):
        self.C, self.kernel, self.shrinking = C, kernel, shrinking

@nice_repr
class DTree:
    def __init__(self, max_depth: DiscreteValue(1, 10)):
        self.max_depth = max_depth

grammar = generate_cfg(Union("Classifier", SVM, DTree))   # "a Classifier is an SVM OR a DTree"
print(grammar, "\n")

s = Sampler(random_state=2)
print("four random configurations:")
for _ in range(4):
    print("   ", grammar.sample(sampler=s))

### The space of pipelines is a graph

Now scale that idea up from a single estimator to a whole pipeline. Given our input and output types and a box of blocks, AutoGOAL can lay out every pipeline that type-checks as a graph: each block is a node, an edge means two blocks fit together, and each node carries its own little space of settings. The nice part is that we can look at this graph, and even draw sample pipelines from it, before running any search at all.

In [ ]:
# --- 3.6  The space of pipelines AutoGOAL searched (under the hood) -------------
# Given our task (INPUT -> OUTPUT) and the registry, AutoGOAL lays out every pipeline
# that type-checks as a graph. The graph you see here IS the space the search explores.
from autogoal.grammar import Sampler
space = build_pipeline_graph(input_types=INPUT, output_type=OUTPUT, registry=registry)
print("blocks in this space:", sorted(a.__name__ for a in space.nodes()))
print("graph:", space.graph.number_of_nodes(), "nodes,", space.graph.number_of_edges(), "edges\n")
print("three pipelines sampled from the space:")
samp = Sampler(random_state=0)
for _ in range(3):
    print("  ", space.sample(sampler=samp), "\n")

### Two dials worth knowing

Two things you will reach for whenever you use this.

The first is the box of blocks itself, because that box is your search space. We handed AutoGOAL a short, curated list. The full `find_classes()` includes everything installed, and some of it (NLTK's sequence-tagging tools, for instance) makes no sense for classifying documents, so on a tight budget the search can waste time on it. Trimming the box to fit the task makes the search faster and usually better.

The second is the metric. By default AutoGOAL optimizes accuracy. If you cared about something else, say macro-F1 on an imbalanced problem, you would pass it as a one-line function. Our four classes are balanced, so it would not change much here, but it is worth knowing where the dial is:

```python
from sklearn.metrics import f1_score
AutoML(..., objectives=lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro"))
```

(AutoGOAL *maximizes* the objective, so higher-is-better metrics like macro-F1 drop straight in; a loss would need a minus sign.)

### A reusable search: budget and logging baked in

We'll run a few more searches, so let's bundle the fixed parts (our task signature, a time budget, a progress log) into one helper. From here on a search is a one-liner: hand it a box of blocks and the data. The budget is **always set**, so no search can run away on us in a live session.

In [ ]:
# --- 3.7  run_search: the fixed parts in one place -------------------------------
from autogoal.ml import AutoML
from autogoal.search import ConsoleLogger   # plain scrolling log -- robust on flaky wifi
                                            # (swap in RichLogger() for the live table)
def run_search(reg, X, y, *, minutes=1.5, pop_size=10, eval_timeout=20, **overrides):
    am = AutoML(
        input=INPUT, output=OUTPUT, registry=reg,
        search_iterations=10_000,                 # the clock is the real stop
        search_timeout=int(minutes * 60),         # total budget (always set)
        evaluation_timeout=eval_timeout,          # cap any single pipeline
        pop_size=pop_size, cross_validation_steps=1,
        errors="ignore", random_state=0, **overrides,
    )
    am.fit(X, y, logger=ConsoleLogger())
    return am

**Your turn: build your own box (≈ 3 min).** This is the knob that matters most: the registry *is* the search space. The rule for a valid box is simple: you need **at least one vectorizer and at least one classifier** (so there's a path from text to a label). The block names from 3.1 are already imported, so you can use them directly.

Fill in `my_registry`, run the cell to validate it with `show_registry`, and fix it until it reports a valid space. We'll run **one search together** afterward. (Fast finishers, as a take-home: find the *smallest* box that still reaches ≥ 85%.)

<details><summary>stuck? one box that works</summary>

```python
my_registry = [TfidfVectorizer, MultinomialNB, ComplementNB, LinearSVC]
```
</details>

In [ ]:
# 🟢 YOUR TURN: fill the box (a vectorizer + a classifier), then validate.
# The names below are already imported (from 3.1 / 3.3) -- just list the ones you want.
my_registry = [
    # e.g.  TfidfVectorizer, MultinomialNB,        # <- needs a vectorizer AND a classifier
]

show_registry(my_registry)        # instant check -- fix the box until it reports a valid space
# We'll run ONE search together after this:
# run_search(my_registry, train.data, y_train, minutes=1)

In [ ]:
# --- 💾 CHECKPOINT 2, restart point for Part 4 (instant) ----------------------
# Jumping in here (or recovered from a restart)? This reloads everything Part 4 needs,
# including INPUT/OUTPUT and run_search, so the rest runs standalone.
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1"); os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
from sklearn.datasets import fetch_20newsgroups
from autogoal.ml import AutoML
from autogoal.search import ConsoleLogger
from autogoal.kb import Seq, Sentence, Supervised, VectorCategorical
from autogoal_contrib import find_classes

CATEGORIES = ['sci.med', 'sci.space', 'rec.autos', 'talk.politics.misc']
strip = ('headers', 'footers', 'quotes')
train = fetch_20newsgroups(subset='train', categories=CATEGORIES, remove=strip, random_state=0)
test  = fetch_20newsgroups(subset='test',  categories=CATEGORIES, remove=strip, random_state=0)
y_train = [train.target_names[i] for i in train.target]
y_test  = [train.target_names[i] for i in test.target]

INPUT  = (Seq[Sentence], Supervised[VectorCategorical])
OUTPUT = VectorCategorical
# (run_search re-defined here so Part 4 stands alone -- identical to 3.7)
def run_search(reg, X, y, *, minutes=1.5, pop_size=10, eval_timeout=20, **overrides):
    am = AutoML(input=INPUT, output=OUTPUT, registry=reg, search_iterations=10_000,
                search_timeout=int(minutes*60), evaluation_timeout=eval_timeout,
                pop_size=pop_size, cross_validation_steps=1, errors="ignore",
                random_state=0, **overrides)
    am.fit(X, y, logger=ConsoleLogger()); return am
print(f"Checkpoint 2 ready, {len(train.data)} train / {len(test.data)} test docs loaded.")

## Part 4: Teaching AutoGOAL something new

This is the part I enjoy most. That box of blocks is not fixed, it is just a list of classes, so you can add your own.

A block is a small class that says two things: what it takes in and gives back (through the types on its `run` method), and what settings it has (through the annotations on its `__init__`). That is the whole contract, no registration step, no plugin system. Let's write a tiny one to see the shape of it, and then wrap a real pretrained transformer as a block and see whether the search wants it.

### A first, tiny block

The smallest thing that counts as a block: take a sentence, hand back a list of words. We subclass `AlgorithmBase`, put types on `run`, and annotate `__init__` with the settings we would like searched. AutoGOAL works out the rest by looking at it.

In [ ]:
# --- 4.1  The smallest possible custom block: a typed transform ----------------
from autogoal.kb import AlgorithmBase, Sentence, Seq, Word
from autogoal.grammar import BooleanValue, DiscreteValue
from autogoal.utils import nice_repr

@nice_repr
class LongWordTokenizer(AlgorithmBase):
    """Split a sentence into words, optionally lowercasing and dropping short tokens."""
    def __init__(self, min_length: DiscreteValue(0, 5), lower: BooleanValue()):
        self.min_length = min_length      # <- searchable: an integer in [0, 5]
        self.lower = lower                # <- searchable: True / False

    def run(self, input: Sentence) -> Seq[Word]:     # <- Sentence in, Seq[Word] out
        if self.lower:
            input = input.lower()
        return [w for w in input.split() if len(w) > self.min_length]

print("declared contract:", LongWordTokenizer.input_types(), "->", LongWordTokenizer.output_type())
print("example:", LongWordTokenizer(min_length=3, lower=True).run("The Apollo program reached the Moon"))

### Wrapping a transformer

Now the interesting one. We take a small pretrained sentence model, MiniLM, and wrap it as a block that turns text into dense vectors. Its `run` takes sentences and returns a matrix, exactly the shape a classifier expects, so once it is in the box the search can pick it or leave it, just like any other block.

(A note on words: MiniLM is an *encoder* (it turns text into vectors), not a generative model like the LLMs you usually hear about. An encoder is *literally* a block, as we're about to see. A generative model can also be *wrapped* as a block (as a feature extractor, or as a judge that scores text), but that takes more plumbing: you decide its output type and how to parse it. The type contract is the same idea; the wrapping is just more work.)

In [ ]:
# --- 4.2  Wrap a transformer LM as an AutoGOAL block --------------------------
import numpy as np
from autogoal.kb import AlgorithmBase, Seq, Sentence, MatrixContinuousDense
from autogoal.grammar import BooleanValue
from autogoal.utils import nice_repr

@nice_repr
class TransformerEmbedding(AlgorithmBase):
    """Pretrained MiniLM sentence embeddings as features: Seq[Sentence] -> (n × d) dense
    matrix (d = the model's embedding size: 384 for MiniLM, 768 for mpnet)."""
    _MODEL = None
    _NAME  = "all-MiniLM-L6-v2"
    _VEC   = {}   # cache ONE raw vector per unique document, so any subset is a cache hit

    def __init__(self, normalize: BooleanValue()):
        self.normalize = normalize

    @classmethod
    def _model(cls):
        if cls._MODEL is None:
            import torch
            from sentence_transformers import SentenceTransformer
            device = "cuda" if torch.cuda.is_available() else "cpu"   # use the vast.ai GPU if present
            cls._MODEL = SentenceTransformer(cls._NAME, device=device)
        return cls._MODEL

    def run(self, input: Seq[Sentence]) -> MatrixContinuousDense:
        # encode only documents we haven't seen; warming the full corpus once (in 4.2b) means
        # the forked search workers inherit a full cache and never re-load the model.
        missing = [d for d in input if d not in TransformerEmbedding._VEC]
        if missing:
            V = self._model().encode(missing, batch_size=64, show_progress_bar=False,
                                     convert_to_numpy=True).astype("float32")
            for d, v in zip(missing, V):
                TransformerEmbedding._VEC[d] = v
        raw = np.vstack([TransformerEmbedding._VEC[d] for d in input])
        if self.normalize:
            n = np.linalg.norm(raw, axis=1, keepdims=True); n[n == 0] = 1.0
            return raw / n
        return raw

print("contract:", TransformerEmbedding.input_types(), "->", TransformerEmbedding.output_type())

> **🔮 Predict the crossover.** We're about to pit MiniLM embeddings against plain TF-IDF as we feed both more and more labelled data. With very little data the pretrained model should win; with a lot, TF-IDF tends to catch up. **At how many training documents do you think TF-IDF overtakes MiniLM?** Write down a number, then run the cell, and we'll plot the two curves so the crossover is easy to see.

In [ ]:
# --- 4.2b  Representation showdown, a learning curve (a *fair* comparison) --------
# Same classifier (LogisticRegression) on both representations, same held-out split,
# averaged over 3 random subsamples -- only the FEATURES differ, as we vary how much
# labelled data we have. (You can skip reading the val_acc helper; the payoff is the
# table + plot.)
import numpy as np, torch
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

y_all = [train.target_names[i] for i in train.target]
print(f"encoding {len(train.data)} docs with MiniLM on "
      f"{'cuda' if torch.cuda.is_available() else 'cpu'} (once)...", flush=True)
EMB_ALL = TransformerEmbedding(normalize=True).run(list(train.data))   # encode all docs once

def val_acc(n, seed):                      # <- plumbing; skip on first read
    idx = train_test_split(np.arange(len(train.data)), train_size=n,
                           random_state=seed, stratify=y_all)[0]
    y = np.array([y_all[i] for i in idx])
    tr, va = train_test_split(np.arange(n), test_size=0.25, random_state=seed, stratify=y)
    emb = LogisticRegression(max_iter=2000).fit(EMB_ALL[idx[tr]], y[tr]).score(EMB_ALL[idx[va]], y[va])
    docs = [train.data[i] for i in idx]
    tf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), sublinear_tf=True)
    Xtr = tf.fit_transform([docs[j] for j in tr]); Xva = tf.transform([docs[j] for j in va])
    tfidf = LogisticRegression(max_iter=2000).fit(Xtr, y[tr]).score(Xva, y[va])
    return emb, tfidf

sizes = (200, 400, 800, 1500, 2200)
emb_mean, tfidf_mean = [], []
print(f"\n{'train docs':>10} | {'MiniLM-emb':>11} | {'TF-IDF':>9} | winner")
print("-" * 48)
for n in sizes:
    es, ts = zip(*[val_acc(n, s) for s in (0, 1, 2)])     # average over 3 subsamples
    e, t = np.mean(es), np.mean(ts)
    emb_mean.append(e); tfidf_mean.append(t)
    print(f"{n:>10} | {e:>10.1%} | {t:>8.1%} | {'MiniLM' if e > t else 'TF-IDF'}")

# the picture: where do the two lines cross?
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 4))
    plt.plot(sizes, [v*100 for v in emb_mean],   "o-", label="MiniLM embeddings")
    plt.plot(sizes, [v*100 for v in tfidf_mean], "s-", label="TF-IDF")
    plt.xlabel("labelled training documents"); plt.ylabel("validation accuracy (%)")
    plt.title("Pretrained embeddings vs TF-IDF as data grows")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", type(e).__name__, e, ")")

Look at where the two lines cross. With only ~200 labelled documents the transformer is far ahead (~89% vs ~67%): it shows up already knowing the language. As both get more data, TF-IDF, which starts from nothing, catches up, and around a couple of thousand documents the two roughly converge, with TF-IDF reaching the top of the pack by ~2,200.

The exact crossover point is noisy: these curves wobble a point or two between runs (you may even see a dip around 400), so read the *trend*, not a single pixel. But the trend is the whole intuition behind transfer learning in one picture: when labelled data is scarce, a pretrained model wins; when there's plenty, simple features can catch up. It turns the vague question "should I use a big model here?" into a concrete one: *how much labelled data do you have?*

(To keep it fair, the same classifier scored both sides on the same held-out split, averaged over a few subsamples, so it's the features talking, not luck.)

In [ ]:
# --- 4.3  Let AutoGOAL choose, with the transformer in the registry -----------
from autogoal_sklearn import (TfidfVectorizer, CountVectorizer,
                              LinearSVC, LogisticRegression, SGDClassifier)

# A low-data slice (800 docs) -- the regime where, above, the transformer was ahead.
N  = 800
Xs = list(train.data[:N])
ys = [train.target_names[i] for i in train.target][:N]

reg_llm = [TransformerEmbedding, TfidfVectorizer, CountVectorizer,
           LinearSVC, LogisticRegression, SGDClassifier]   # classic blocks AND your block
print("registry now includes your block:",
      [getattr(c, "__name__", str(c)) for c in reg_llm])

# one line, budget + logging already baked in:
automl_llm = run_search(reg_llm, Xs, ys, minutes=2, eval_timeout=40)
print("\nbest validation score:", automl_llm.best_scores_)
print("AutoGOAL picked:\n", automl_llm.best_pipelines_[0])

Did the search keep your transformer? On our runs it did, pairing `TransformerEmbedding` with a linear classifier (a strong validation score, in the high 0.8s), which is what the curve said should win at this size. It's a stochastic search, so the exact pairing and score will move around for you. (We report the *validation* score here by design: we already spent our one honest look at the test set earlier.)

But the real point is what just happened. We added one small class, and AutoGOAL weighed a pretrained transformer against the classic approaches on equal terms and told us which was worth using here. That's the bridge from classical AutoML to the LLM era, with one honest caveat: an *encoder* like MiniLM is literally just another block. A *generative* LLM can be wrapped as one too (a featurizer, or a judge that scores candidates), but that wrapping is real work: you choose its output type and how to parse it. The idea carries over; the plumbing grows.

**Your turn, after class.** A few things worth trying on your own:

1. Swap the model: point `_NAME` at `"all-mpnet-base-v2"` (stronger, 768 dimensions) and re-run the curve. At home that is about a 420 MB download, so you would first unset the offline flags (`os.environ.pop("HF_HUB_OFFLINE", None)`) or pre-cache it in `setup.sh`.
2. Add a setting: give `TransformerEmbedding` a pooling choice (`CategoricalValue("mean", "max")`) and use it in `run`.
3. Write a little featurizer of your own, say counts of a few domain keywords, and add it to the box.

### One more thing, if you are curious: your own type

The types are not fixed either. You can carve out a new one by subclassing an existing type and tightening what it matches. Here is an `Email` type that only accepts strings with an `@` in them; a block that asked for `Seq[Email]` would then turn down anything else.

In [ ]:
# ★ A custom semantic type, in five lines
from autogoal.kb import Text

class Email(Text):                       # Email ⊂ Text, automatically
    @classmethod
    def _match(cls, x):
        return super()._match(x) and "@" in x

print("isinstance('a@b.com', Email):", isinstance('a@b.com', Email))
print("isinstance('hello',   Email):", isinstance('hello', Email))
print("issubclass(Email, Text)     :", issubclass(Email, Text))

## Where we landed

We climbed a ladder, and at every rung the machine did more of the work. We built a classifier by hand and improved it with a few good choices. We packaged it as a `Pipeline`. We handed the whole pipeline to AutoGOAL and let it choose the blocks *and* their settings. And then we widened the box with a block of our own and let the search decide whether a transformer was worth it.

The idea underneath all of it is simple: AutoML is search over typed pipelines. You stay in charge of four things (the blocks you offer, the budget you allow, the metric you care about, and where the search may look), and you hand off the tedious part. And adding an LLM mostly fits this same frame: an encoder is just another block, and even a generative model can be wrapped as one, with the search still reasoning over typed pipelines; you just do more of the wrapping.

Two ways to keep going. The **appendix** below works through classical grid and random search, the systematic version of the hand-tuning we opened with. Or push the search itself: give AutoGOAL more time, hand it the full set of blocks, or point it at your own data (`AutoML.fit` takes any list of texts and labels). Thanks for following along.

<a id="appendix"></a>
## Appendix: Classical HPO with grid & random search

Earlier we tuned the pipeline by hand and reached ~84%. This appendix shows the *systematic* way scikit-learn offers to do the same job: define a space of settings and let a search sweep it. It's the classical-HPO foundation underneath everything, and it lands on essentially the configuration we reasoned our way to, without the guesswork.

Run it after the notebook above (it reuses `train`/`test`), or read it on your own time. We deliberately moved it here so the session could spend its minutes on the AutoML payoff.

In [ ]:
# --- A.1  Count the search space ----------------------------------------------
from math import prod
knobs = {
    'ngram_range':  [(1,1), (1,2), (1,3)],
    'min_df':       [1, 2, 5],
    'max_df':       [0.5, 0.75, 1.0],
    'stop_words':   [None, 'english'],
    'sublinear_tf': [False, True],
    'C':            [0.03, 0.1, 0.3, 1, 3, 10],
}
print("configurations in a full grid:", prod(len(v) for v in knobs.values()))
print("...and that is before we even consider a different classifier.")

That is 648 combinations, and this is still a small grid. Grid search tries every combination and keeps the best by cross-validation. To keep it quick, we'll search a representative **120-point slice**.

> **🔮 Predict first.** The baseline was 72.1%. Before you run it, commit to a number: how much will tuning buy us? 75%? 80%? 85%? Write it down.

In [ ]:
# --- A.2  Grid search: try a representative slice (120 of the 648), keep best by CV ---
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([('vec', TfidfVectorizer()),
                 ('clf', LinearSVC(max_iter=5000, random_state=0))])

# The search space is a dict: 'stepname__hyperparameter' -> values to try (the same
# double-underscore naming from the Pipeline section). 'vec__' settings go to the
# vectorizer step, 'clf__' to the classifier -- this dict *is* the simplest way to say it.
grid = {'vec__ngram_range':  [(1,1), (1,2)],
        'vec__min_df':       [1, 2, 5],
        'vec__stop_words':   [None, 'english'],
        'vec__sublinear_tf': [False, True],
        'clf__C':            [0.1, 0.3, 1, 3, 10]}

gs = GridSearchCV(pipe, grid, cv=5, n_jobs=-1).fit(train.data, train.target)
print(f"grid: {len(gs.cv_results_['params'])} configs × 5-fold CV")
print(f"best cross-val accuracy = {gs.best_score_:.1%}")
print(f"TEST accuracy           = {gs.score(test.data, test.target):.1%}")
print("winning knobs:", gs.best_params_)

How close was your guess? We went from 72.1% to **84.1%** on the test set, just by switching to TF-IDF, allowing word pairs, dropping English stop-words, and tuning the regularization. Notice cross-validation scored a little higher than the test set; it usually does, because we kept the *best of many* configs on the validation signal, and "best of many" is optimistic, so the untouched test set is the honest check.

The catch with grids is that they grow fast. This small one was already 648 points, and one more knob would push it into the thousands. Random search gets around that by sampling combinations instead of listing them all. Bergstra and Bengio showed back in 2012 that this often finds a good model sooner, because only a handful of knobs really matter, and random sampling does not waste time on the rest.

In [ ]:
# --- A.3  Random search: sample the space instead of enumerating it -----------
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

# Same step__param dict, with one upgrade: a continuous knob can be a *distribution*
# to sample from, not just a fixed list. (We reuse `pipe` from A.2.)
space = {'vec__ngram_range':  [(1,1), (1,2), (1,3)],
         'vec__min_df':       [1, 2, 3, 5],
         'vec__max_df':       [0.5, 0.75, 1.0],
         'vec__stop_words':   [None, 'english'],
         'vec__sublinear_tf': [False, True],
         'clf__C':            loguniform(1e-2, 1e2)}      # a distribution, not a list

rs = RandomizedSearchCV(pipe, space, n_iter=40, cv=5, random_state=0, n_jobs=-1)
rs.fit(train.data, train.target)
print(f"random: 40 sampled configs × 5-fold CV")
print(f"best cross-val accuracy = {rs.best_score_:.1%}")
print(f"TEST accuracy           = {rs.score(test.data, test.target):.1%}")
print("winning knobs:", {k: (round(v,3) if isinstance(v,float) else v) for k,v in rs.best_params_.items()})

Forty random tries matched the grid for a fraction of the work: the case for sampling rather than enumerating (Bergstra & Bengio, 2012: only a few knobs really matter, so random sampling doesn't waste effort on the rest). Both land within a whisker of the 84% we reached by hand, which is the real lesson of classical HPO: it **systematises** the manual tuning we opened with, and takes the guesswork out.

What it does *not* do is question the shape of the pipeline: it still assumes a vectorizer then a classifier, with the blocks you picked. Letting the search choose the steps themselves is exactly the leap we took with AutoGOAL.

**Your turn** (a quick one, let's do it together). Right now the classifier is fixed. Let's make the choice of classifier part of the search as well. In the cell below, compare the linear SVM against logistic regression and multinomial naive Bayes. (One catch: naive Bayes needs non-negative features, so keep TF-IDF.) Which one comes out ahead here?

In [ ]:
# 🟢 YOUR TURN 1: make the *classifier* part of the search.
# GridSearchCV can vary a whole pipeline step, not just its numbers: put a LIST of
# classifier instances under the key 'clf' and it will try each one and rank them.
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

pipe2 = Pipeline([('vec', TfidfVectorizer(stop_words='english', ngram_range=(1,2))),
                  ('clf', LinearSVC(max_iter=5000, random_state=0))])

grid2 = {
    # TODO: 'clf': [ ...three classifiers to compare... ]   # NB needs TF-IDF (we have it)
    # TODO: optionally one vectorizer knob, e.g. 'vec__sublinear_tf': [False, True]
}

# Uncomment once grid2 is filled in:
# gs2 = GridSearchCV(pipe2, grid2, cv=5, n_jobs=-1).fit(train.data, train.target)
# print(f"best CV {gs2.best_score_:.1%} with {type(gs2.best_params_['clf']).__name__}")
# assert gs2.best_score_ > 0.83, "hint: did you add all three classifiers?"

Give it a try first. When you're ready, run the next cell to see one answer, and the full ranking of the three classifiers.

In [ ]:
# Solution to Your turn 1
grid2 = {
    'clf': [LinearSVC(max_iter=5000, random_state=0), LogisticRegression(max_iter=2000), MultinomialNB()],
    'vec__sublinear_tf': [False, True],
}
gs2 = GridSearchCV(pipe2, grid2, cv=5, n_jobs=-1).fit(train.data, train.target)
print(f"best CV = {gs2.best_score_:.1%}  with  {type(gs2.best_params_['clf']).__name__}")
print("full ranking:")
import numpy as np
for acc, p in sorted(zip(gs2.cv_results_['mean_test_score'], gs2.cv_results_['params']), reverse=True):
    print(f"  {acc:.1%}  {type(p['clf']).__name__:20s} sublinear_tf={p['vec__sublinear_tf']}")